基於時序資料之桌球戰術與結果預測競賽

In [2]:
import pandas as pd
import numpy as np

# 1. 讀取資料
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 2. 核心特徵選取
# 這些是每一拍會變動的特徵 (Time-step features)
features = ['strikeId', 'handId', 'strengthId', 'spinId', 'positionId', 'actionId']
target = ['actionId','pointId','serverGetPoint'] # 假設我們要預測下一拍的動作

# 3. 依照 rally_uid 進行分組轉換
def create_sequences(df, max_len=15):
    sequences = []
    labels = []

    for uid, group in df.groupby('rally_uid'):
        # 確保依擊球順序排列
        group = group.sort_values('strikeNumber')

        # 提取特徵矩陣
        feat_matrix = group[features].values

        # 建立滑動窗口 (例如用前 3 拍預測第 4 拍)
        for i in range(1, len(feat_matrix)):
            seq = feat_matrix[:i] # 前 i 拍
            # 補零到固定長度 max_len
            if len(seq) < max_len:
                pad = np.zeros((max_len - len(seq), len(features)))
                seq = np.vstack((pad, seq))
            else:
                seq = seq[-max_len:]

            sequences.append(seq)
            labels.append(group.iloc[i][target]) # 第 i+1 拍的動作

    return np.array(sequences), np.array(labels)

# 執行轉換 (這步會花一點時間)
# X_train, y_train = create_sequences(train_df)

In [3]:
import os
print("目前目錄下的檔案有：", os.listdir('/content/'))

目前目錄下的檔案有： ['.config', 'test.csv', 'train.csv', 'sample_data']


In [4]:
import pandas as pd
import os

# 1. 自動檢查 Colab 目前有哪些檔案
current_files = os.listdir('/content/')
print(f"目前目錄下的檔案：{current_files}")

# 2. 嘗試用正確的路徑讀取 (加上 /content/ 前綴)
try:
    # 這裡確保路徑是 Colab 的預設路徑
    train = pd.read_csv('/content/train.csv')
    print("✅ 成功讀取 train.csv！")

    # 3. 執行你剛才想跑的分析
    print("\n--- 每一個回合(Rally)的拍數統計 ---")
    strike_counts = train.groupby('rally_uid')['strikeNumber'].max()
    print(strike_counts.value_counts().sort_index())

except FileNotFoundError:
    print("❌ 錯誤：找不到檔案！請確認左側資料夾選單中，檔名是否真的叫 'train.csv'")
except Exception as e:
    print(f"❌ 發生其他錯誤：{e}")

目前目錄下的檔案：['.config', 'test.csv', 'train.csv', 'sample_data']
✅ 成功讀取 train.csv！

--- 每一個回合(Rally)的拍數統計 ---
strikeNumber
2     1869
3     2585
4     3030
5     1933
6     1598
7      974
8      793
9      520
10     379
11     287
12     221
13     158
14     135
15      93
16      65
17      55
18      55
19      46
20      22
21      26
22      18
23      27
24      20
25       7
26      12
27       8
28       4
29       8
30       5
31       3
32       5
33       4
34       2
35       6
36       3
37       4
38       2
39       2
40       5
42       3
45       1
46       1
52       1
Name: count, dtype: int64


In [5]:
# 範例：將資料按 rally 排序
train = train.sort_values(['rally_uid', 'strikeNumber'])

# 找出所有的類別數量 (用於 Embedding)
num_actions = train['actionId'].nunique()
num_positions = train['positionId'].nunique()
num_getpoint = train['serverGetPoint'].nunique()

print(f"總共有 {num_actions} 種動作，{num_positions} 種落點方位。{num_getpoint} 種得分")

總共有 19 種動作，4 種落點方位。2 種得分


第一步：構建訓練資料 (X, y)
我們需要把 train.csv 的每一列轉成「序列」，並提取每個 rally_uid 的最終結果作為標籤。

In [9]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 設定參數
MAX_SEQ_LEN = 10  # 每個回合考慮前 10 拍，不足補 0
FEATURE_COLS = ['strikeNumber', 'scoreSelf', 'scoreOther', 'handId', 'spinId', 'positionId', 'serverGetPoint']

def prepare_data_v5(df, max_len=10):
    X = []
    y_action = []
    y_point = []
    y_server = []
    uids = [] # 儲存 rally_uid，預測時會用到

    # 根據你的想法剔除不具參考性的特徵 (例如 sex, match)
    # 保留物理與戰術相關特徵
    feature_cols = [
        'strikeNumber', 'scoreSelf', 'scoreOther', 'serverGetPoint',
        'handId', 'strengthId', 'spinId', 'positionId'
    ]

    for uid, group in df.groupby('rally_uid'):
        group = group.sort_values('strikeNumber')
        feat_matrix = group[feature_cols].values

        # 滑動窗口：i 代表我們要預測的那一球 (第 N 球)
        # 從 i=1 開始，代表至少有 1 球 (第 0 球) 當作歷史資料
        for i in range(1, len(feat_matrix)):
            # 取前 i 球的特徵 (第 0 到 i-1 球)
            seq = feat_matrix[:i]

            # Padding
            if len(seq) < max_len:
                pad = np.zeros((max_len - len(seq), len(feature_cols)))
                seq = np.vstack((pad, seq))
            else:
                seq = seq[-max_len:]

            X.append(seq)

            # 儲存第 i 球 (第 N 球) 的預測目標
            row = group.iloc[i]
            y_action.append(row['actionId'])
            y_point.append(row['pointId'])
            y_server.append(row['serverGetPoint'])
            uids.append(uid) # 記錄這組資料是屬於哪個 rally

    return (np.array(X),
            np.array(y_action).astype('int32'),
            np.array(y_point).astype('int32'),
            np.array(y_server).astype('float32'),
            uids)

# 執行讀取
X_train, y_act, y_ptr, y_srv, train_uids = prepare_data_v5(train)
print(f"新的訓練樣本數: {len(X_train)}, {len(y_act)}, {len(y_ptr)}, {len(y_srv)}, {len(train_uids)}")

新的訓練樣本數: 69712, 69712, 69712, 69712, 69712


第二步：建立多輸出模型 (Keras Functional API)
這段程式碼會建立一個 LSTM 模型，最後分出三個輸出層：

In [17]:
from re import X
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split

num_classes = 10

# 重新定義模型
inputs = Input(shape=(10, 8))
x = LSTM(64, return_sequences=True)(inputs)
x = Dropout(0.2)(x)
x = LSTM(32)(x)
x = Dense(64, activation='relu')(x)
# 最終輸出：第 N 球的落點機率
outputs = Dense(num_classes, activation='softmax')(x)

# --- 關鍵：明確命名輸出層 ---
# Action (動作0-18 -> 19個輸出)
out_action = Dense(17, activation='softmax', name='act_head')(x)
# Point (落點 0-9 -> 10個輸出)
out_point = Dense(11, activation='softmax', name='ptr_head')(x)
# Server (是否得分 0-1 -> 1個輸出)
out_server = Dense(1, activation='sigmoid', name='srv_head')(x)

model_multi = Model(inputs=inputs, outputs=[out_action, out_point, out_server])

model_multi.compile(
    optimizer='adam',
    loss={
        'act_head': 'sparse_categorical_crossentropy',
        'ptr_head': 'sparse_categorical_crossentropy',
        'srv_head': 'binary_crossentropy'
    },
    # 這裡也要改成字典，對應三個輸出的名稱
    metrics={
        'act_head': 'accuracy',
        'ptr_head': 'accuracy',
        'srv_head': 'accuracy'
    },
    loss_weights={'act_head': 1.0, 'ptr_head': 1.2, 'srv_head': 0.8}
)

# 1. 手動切分資料 (解決 NameError)
# 我們把 prepare_data_v5 產生的 X_train, y_act, y_ptr, y_srv 進行切分
X_train_final, X_val, \
y_act_train, y_act_val, \
y_ptr_train, y_ptr_val, \
y_srv_train, y_srv_val = train_test_split(
    X_train, y_act, y_ptr, y_srv,
    test_size=0.2,    # 撥出 20% 作為驗證集 (模擬考)
    random_state=42   # 固定隨機種子，確保每次切分結果一樣
)

print(f"✅ 資料切分成功！")
print(f"訓練樣本數: {len(X_train_final)}")
print(f"驗證樣本數: {len(X_val)}")





# 使用字典匹配名稱，並傳入手動切好的驗證集
history = model_multi.fit(
    X_train_final,
    {
        'act_head': y_act_train,
        'ptr_head': y_ptr_train,
        'srv_head': y_srv_train
    },
    validation_data=(
        X_val,
        {
            'act_head': y_act_val,
            'ptr_head': y_ptr_val,
            'srv_head': y_srv_val
        }
    ),
    epochs=50,
    batch_size=64,
    verbose=1
)

model_multi.summary()

✅ 資料切分成功！
訓練樣本數: 55769
驗證樣本數: 13943
Epoch 1/50
872/872 ━━━━━━━━━━━━━━━━━━━━ 23s 17ms/step - act_head_accuracy: 0.2593 - act_head_loss: 2.1802 - loss: 4.5788 - ptr_head_accuracy: 0.2558 - ptr_head_loss: 1.9114 - srv_head_accuracy: 0.9355 - srv_head_loss: 0.1304 - val_act_head_accuracy: 0.3164 - val_act_head_loss: 2.0278 - val_loss: 4.2427 - val_ptr_head_accuracy: 0.2818 - val_ptr_head_loss: 1.8435 - val_srv_head_accuracy: 1.0000 - val_srv_head_loss: 0.0032
Epoch 2/50
872/872 ━━━━━━━━━━━━━━━━━━━━ 14s 16ms/step - act_head_accuracy: 0.3284 - act_head_loss: 1.9864 - loss: 4.1952 - ptr_head_accuracy: 0.2969 - ptr_head_loss: 1.8391 - srv_head_accuracy: 0.9999 - srv_head_loss: 0.0027 - val_act_head_accuracy: 0.3429 - val_act_head_loss: 1.9309 - val_loss: 4.0970 - val_ptr_head_accuracy: 0.3194 - val_ptr_head_loss: 1.8042 - val_srv_head_accuracy: 1.0000 - val_srv_head_loss: 0.0014
Epoch 3/50
872/872 ━━━━━━━━━━━━━━━━━━━━ 20s 16ms/step - act_head_accuracy: 0.3525 - act_head_loss: 1.8970 - loss: 4.

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_5       │ (None, 10, 8)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_10 (LSTM)      │ (None, 10, 64)    │     18,688 │ input_layer_5[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 10, 64)    │          0 │ lstm_10[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_11 (LSTM)      │ (None, 32)        │     12,416 │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 64)        │      2,112 │ lstm_11[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ act_head (Dense)    │ (None, 17)        │      1,105 │ dense_10[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ptr_head (Dense)    │ (None, 11)        │        715 │ dense_10[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ srv_head (Dense)    │ (None, 1)         │         65 │ dense_10[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 105,305 (411.35 KB)

 Trainable params: 35,101 (137.11 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 70,204 (274.24 KB)

In [38]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, BatchNormalization

# 根據你 prepare_data_v3 的設定
# MAX_SEQ_LEN = 10, 特徵數 = 7 (包含 positionId)
num_landing_areas = 10 # 預測 0-9 區

inputs = Input(shape=(10, 7))

# 建立 LSTM 特徵提取
x = LSTM(128, return_sequences=True)(inputs) # 資料量變大，神經元可以多一點
x = Dropout(0.3)(x)
x = LSTM(64)(x)
x = BatchNormalization()(x)
x = Dense(64, activation='relu')(x)

# 最終輸出：落點 1-10 的機率
# 使用 softmax 進行多分類
outputs = Dense(num_landing_areas, activation='softmax')(x)

model_landing = Model(inputs=inputs, outputs=outputs)

model_landing.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_landing.summary()

# 確保標籤是整數型態，這能避免 Keras 3 的底層報錯
y_train_fixed = y_train.astype('int32')

history = model_landing.fit(
    X_train,
    y_train_fixed,
    epochs=50,       # 5萬筆資料，可以先跑 50 次
    batch_size=64,   # 批次大小設 64 跑得比較快
    validation_split=0.2, # 拿 20% 來驗證是否過擬合
    verbose=1
)

Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_8 (InputLayer)      │ (None, 10, 7)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_12 (LSTM)                  │ (None, 10, 128)        │        69,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 10, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_13 (LSTM)                  │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 124,106 (484.79 KB)

 Trainable params: 123,978 (484.29 KB)

 Non-trainable params: 128 (512.00 B)

Epoch 1/50
872/872 ━━━━━━━━━━━━━━━━━━━━ 44s 44ms/step - accuracy: 0.2610 - loss: 1.9089 - val_accuracy: 0.2698 - val_loss: 1.8698
Epoch 2/50
872/872 ━━━━━━━━━━━━━━━━━━━━ 34s 39ms/step - accuracy: 0.2733 - loss: 1.8742 - val_accuracy: 0.2752 - val_loss: 1.8606
Epoch 3/50
872/872 ━━━━━━━━━━━━━━━━━━━━ 35s 40ms/step - accuracy: 0.2778 - loss: 1.8620 - val_accuracy: 0.2859 - val_loss: 1.8536
Epoch 4/50
872/872 ━━━━━━━━━━━━━━━━━━━━ 34s 39ms/step - accuracy: 0.2828 - loss: 1.8532 - val_accuracy: 0.2826 - val_loss: 1.8520
Epoch 5/50
872/872 ━━━━━━━━━━━━━━━━━━━━ 36s 41ms/step - accuracy: 0.2841 - loss: 1.8466 - val_accuracy: 0.2830 - val_loss: 1.8524
Epoch 6/50
872/872 ━━━━━━━━━━━━━━━━━━━━ 43s 43ms/step - accuracy: 0.2849 - loss: 1.8410 - val_accuracy: 0.2867 - val_loss: 1.8436
Epoch 7/50
872/872 ━━━━━━━━━━━━━━━━━━━━ 37s 42ms/step - accuracy: 0.2886 - loss: 1.8368 - val_accuracy: 0.2861 - val_loss: 1.8489
Epoch 8/50
872/872 ━━━━━━━━━━━━━━━━━━━━ 35s 40ms/step - accuracy: 0.2875 - loss: 1.8308 - 

處理測試資料

In [12]:
def prepare_test_data(df, max_len=10):
    X = []
    uids = []
    feature_cols = [
        'strikeNumber', 'scoreSelf', 'scoreOther', 'serverGetPoint',
        'handId', 'strengthId', 'spinId', 'positionId'
    ]

    # 按 rally_uid 分組，每一組代表一個測試樣本
    for uid, group in df.groupby('rally_uid'):
        group = group.sort_values('strikeNumber')
        feat_matrix = group[feature_cols].values

        # 我們用這回合現有的所有球路作為輸入
        seq = feat_matrix

        # Padding (補齊到 max_len)
        if len(seq) < max_len:
            pad = np.zeros((max_len - len(seq), len(feature_cols)))
            seq = np.vstack((pad, seq))
        else:
            seq = seq[-max_len:]

        X.append(seq)
        uids.append(uid)

    return np.array(X), uids

# 讀取測試集並轉換
test_df = pd.read_csv('/content/test.csv')
X_test, test_uids = prepare_test_data(test_df)
print(f"測試集樣本數: {len(X_test)}")

測試集樣本數: 1236


進行預測

In [13]:
# 1. 執行預測
# 預測結果會是一個 List，包含 [action_probs, point_probs, server_probs]
predictions = model_multi.predict(X_test)

pred_actions = np.argmax(predictions[0], axis=1) # 取機率最大的 actionId
pred_points = np.argmax(predictions[1], axis=1)  # 取機率最大的 pointId (1-10)
pred_servers = (predictions[2] > 0.5).astype(int).flatten() # serverGetPoint (0 或 1)

# 2. 建立 DataFrame
submission = pd.DataFrame({
    'rally_uid': test_uids,
    'actionId': pred_actions,
    'pointId': pred_points,
    'serverGetPoint': pred_servers
})

# 3. 檢查輸出結果的前幾行
print(submission.head())

39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step


AxisError: axis 1 is out of bounds for array of dimension 1